In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/vihangasathsara-75/cm2604-cw-telco-churn-prediction/refs/heads/main/data/raw/Telco-Churn.csv"
df = pd.read_csv(url)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df = df.drop(columns=["customerID"])
print(df.shape)

(7043, 20)


In [2]:
service_cols = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
                "TechSupport", "StreamingTV", "StreamingMovies"]

for col in service_cols:
    df[col] = df[col].replace({"No internet service": "No", "No phone service": "No"})

for col in service_cols:
    print(col, df[col].unique())

MultipleLines ['No' 'Yes']
OnlineSecurity ['No' 'Yes']
OnlineBackup ['Yes' 'No']
DeviceProtection ['No' 'Yes']
TechSupport ['No' 'Yes']
StreamingTV ['No' 'Yes']
StreamingMovies ['No' 'Yes']


In [3]:
from sklearn.model_selection import train_test_split

y = df["Churn"].map({"No": 0, "Yes": 1})
X = df.drop(columns=["Churn"])
assert y.notnull().all()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print("Churn rate train:", round(y_train.mean(), 3), "test:", round(y_test.mean(), 3))

(5634, 19) (1409, 19)
Churn rate train: 0.265 test: 0.265


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = X.select_dtypes(include="object").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), cat_cols),
    ],
    remainder="passthrough",
    sparse_threshold=0,
)

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "KNN": KNeighborsClassifier(),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

baseline = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    baseline[name] = {m: scores[f"test_{m}"].mean() for m in scoring}

baseline_df = pd.DataFrame(baseline).T.round(3)
baseline_df

,accuracy,precision,recall,f1,roc_auc
KNN,0.770,0.571,0.535,0.552,0.781
LogisticRegression,0.802,0.654,0.543,0.593,0.846
RandomForest,0.788,0.634,0.480,0.546,0.824
